# Codebase Workflow Walkthrough

This notebook is a first guided tour of the live `src/` code.

It is designed to answer three questions clearly:

1. What is the workflow of the code?
2. What does each layer do?
3. What is already implemented and working?

We start from the simplest direct interaction examples and move upward through the model layer, gauge metadata, gauge compilation, and the most recent convention-fixed covariant and pure-gauge features.

## Recommended way to read this notebook

Run the notebook from top to bottom.

The order is intentional:

- direct engine with scalars
- add derivatives
- add fermions and tensor structures
- mix sectors
- understand the metadata layer
- build a first `Model`
- compile gauge interactions from metadata
- compile covariant kinetic terms
- compile pure-gauge Yang-Mills structures

That mirrors the actual growth of the codebase.

In [1]:
import sys
from pathlib import Path
from fractions import Fraction
from pprint import pprint

root = Path.cwd()
src_path = root / "src"
if not src_path.exists():
    src_path = root.parent / "src"

sys.path.insert(0, str(src_path))

from model_symbolica import (
    S,
    Expression,
    I,
    Delta,
    Dot,
    pcomp,
    vertex_factor,
    simplify_deltas,
    simplify_spinor_indices,
    simplify_vertex,
    infer_derivative_targets,
    compact_vertex_sum_form,
    compact_sum_notation,
)

from spenso_structures import (
    SPINOR_KIND,
    LORENTZ_KIND,
    COLOR_FUND_KIND,
    COLOR_ADJ_KIND,
    gamma_matrix,
    gamma5_matrix,
    gauge_generator,
    structure_constant,
    lorentz_metric,
    simplify_gamma_chain,
)

from operators import (
    psi_bar_gamma_psi,
    psi_bar_gamma5_psi,
    current_current,
    quark_gluon_current,
    scalar_gauge_contact,
)

from model import (
    Field,
    GaugeGroup,
    GaugeRepresentation,
    InteractionTerm,
    DerivativeAction,
    Model,
    DiracKineticTerm,
    ComplexScalarKineticTerm,
    GaugeKineticTerm,
    SPINOR_INDEX,
    LORENTZ_INDEX,
    COLOR_FUND_INDEX,
    COLOR_ADJ_INDEX,
)

from gauge_compiler import (
    compile_minimal_gauge_interactions,
    compile_covariant_terms,
)

print("Python:", sys.executable)
print("src path:", src_path.resolve())

x = S("x")
d = S("d")

def direct_vertex(species_map=None, simplify_gamma=False, strip_externals=True, include_delta=False, **kwargs):
    expr = vertex_factor(
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
        **kwargs,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr

def model_vertex(interaction, external_legs, species_map=None, strip_externals=True, include_delta=False, simplify_gamma=False):
    expr = vertex_factor(
        interaction=interaction,
        external_legs=external_legs,
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr

def show(title, expr):
    print(f"\
=== {title} ===")
    print(expr)

def labels(terms):
    return [term.label for term in terms]

Python: /Users/rems/Library/CloudStorage/OneDrive-ETHZurich/ETHz/ETHz_FS26/MScThesis/.venv/bin/python
src path: /Users/rems/Library/CloudStorage/OneDrive-ETHZurich/ETHz/ETHz_FS26/MScThesis/src


## 0. Workflow in one paragraph

The code has one central engine: `vertex_factor(...)` in `src/model_symbolica.py`.

Everything else either:

- prepares inputs for that engine, or
- builds symbolic tensor structures for the engine, or
- compiles higher-level physics declarations into ordinary `InteractionTerm` objects that the same engine can evaluate.

So the notebook always asks the same question: **what do we feed into `vertex_factor(...)`, and how do we build those inputs more systematically?**

In [8]:
p1, p2, p3, p4 = S("p1", "p2", "p3", "p4")
b1, b2, b3, b4 = S("b1", "b2", "b3", "b4")

phi0, chi0 = S("phi0", "chi0")
phiC0, phiCdag0 = S("phiC0", "phiCdag0")
phiQED0, phiQEDdag0 = S("phiQED0", "phiQEDdag0")
phiQCD0, phiQCDdag0 = S("phiQCD0", "phiQCDdag0")
psi0, psibar0 = S("psi0", "psibar0")
psiQED0, psibarQED0 = S("psiQED0", "psibarQED0")
A0, G0 = S("A0", "G0")

mu, nu, rho, sigma = S("mu", "nu", "rho", "sigma")
mu3, mu4 = S("mu3", "mu4")

lam4, g_sym, gD, yF, gV, eQED, qPsi, qPhi, gS = S(
    "lam4", "g", "gD", "yF", "gV", "eQED", "qPsi", "qPhi", "gS"
)

alpha_s = S("alpha_s")
a = S("a")
i_bar, i_psi = S("i_bar", "i_psi")
i1, i2, i3, i4 = S("i1", "i2", "i3", "i4")
s1, s2, s3, s4 = S("s1", "s2", "s3", "s4")
a3, a4, a5, a6 = S("a3", "a4", "a5", "a6")
c1, c2 = S("c1", "c2")

print("Common symbols initialised.")

Common symbols initialised.


## 1. Direct engine input with pure scalars

The most basic interface is the **direct API**.

We provide:

- `coupling`
- `alphas`: the field species appearing in the interaction monomial
- `betas`: the species attached to the external legs
- `ps`: the external momenta

For bosons this is already enough for the combinatorics.

In [9]:
L_phi4 = {
    "coupling": lam4,
    "alphas": [phi0, phi0, phi0, phi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
}

L_phi2chi2 = {
    "coupling": g_sym,
    "alphas": [phi0, phi0, chi0, chi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
}

print("Direct API dictionary for phi^4:")
pprint(L_phi4)

V_phi4_full = vertex_factor(x=x, d=d, **L_phi4)
V_phi4_reduced = direct_vertex(
    **L_phi4,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_phi2chi2 = direct_vertex(
    **L_phi2chi2,
    species_map={b1: phi0, b2: phi0, b3: chi0, b4: chi0},
)

show("phi^4 with overall momentum delta", V_phi4_full)
show("phi^4 reduced vertex", V_phi4_reduced)
show("phi^2 chi^2 reduced vertex", V_phi2chi2)

Direct API dictionary for phi^4:
{'alphas': [phi0, phi0, phi0, phi0],
 'betas': [b1, b2, b3, b4],
 'coupling': lam4,
 'ps': [p1, p2, p3, p4]}
=== phi^4 with overall momentum delta ===
24𝑖*lam4*(2*𝜋)^d*delta(b1,phi0)*delta(b2,phi0)*delta(b3,phi0)*delta(b4,phi0)*Delta(p1+p2+p3+p4)
=== phi^4 reduced vertex ===
24𝑖*lam4
=== phi^2 chi^2 reduced vertex ===
4𝑖*g


## 2. Add derivatives

Derivative interactions are encoded by:

- `derivative_indices`: which Lorentz indices appear
- `derivative_targets`: which field slots those derivatives act on

The helper `infer_derivative_targets(...)` builds these lists from a more readable slot-based description.

In [10]:
deriv_indices, deriv_targets = infer_derivative_targets([(0, [mu]), (1, [nu])])

L_deriv = {
    "coupling": gD,
    "alphas": [phi0, phi0, phi0, phi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
    "derivative_indices": deriv_indices,
    "derivative_targets": deriv_targets,
}

V_deriv = direct_vertex(
    **L_deriv,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_deriv_compact = compact_vertex_sum_form(
    coupling=gD,
    ps=[p1, p2, p3, p4],
    derivative_indices=deriv_indices,
    derivative_targets=deriv_targets,
    d=d,
    field_species=[phi0, phi0, phi0, phi0],
    leg_species=[phi0, phi0, phi0, phi0],
)

print("Derivative indices:", deriv_indices)
print("Derivative targets:", deriv_targets)
print("Compact sigma notation:", compact_sum_notation(derivative_indices=deriv_indices, derivative_targets=deriv_targets, n_legs=4))

show("Expanded derivative vertex", V_deriv)
show("Compact derivative sum form", V_deriv_compact)

Derivative indices: [mu, nu]
Derivative targets: [0, 1]
Compact sigma notation: (2)! * Σ_{a, b distinct} p_{a,mu} p_{b,nu}
=== Expanded derivative vertex ===
1𝑖*(-2*gD*pcomp(p1,mu)*pcomp(p2,nu)-2*gD*pcomp(p1,mu)*pcomp(p3,nu)-2*gD*pcomp(p1,mu)*pcomp(p4,nu)-2*gD*pcomp(p1,nu)*pcomp(p2,mu)-2*gD*pcomp(p1,nu)*pcomp(p3,mu)-2*gD*pcomp(p1,nu)*pcomp(p4,mu)-2*gD*pcomp(p2,mu)*pcomp(p3,nu)-2*gD*pcomp(p2,mu)*pcomp(p4,nu)-2*gD*pcomp(p2,nu)*pcomp(p3,mu)-2*gD*pcomp(p2,nu)*pcomp(p4,mu)-2*gD*pcomp(p3,mu)*pcomp(p4,nu)-2*gD*pcomp(p3,nu)*pcomp(p4,mu))
=== Compact derivative sum form ===
-1𝑖*gD*(2*𝜋)^d*(2*pcomp(p1,mu)*pcomp(p2,nu)+2*pcomp(p1,mu)*pcomp(p3,nu)+2*pcomp(p1,mu)*pcomp(p4,nu)+2*pcomp(p1,nu)*pcomp(p2,mu)+2*pcomp(p1,nu)*pcomp(p3,mu)+2*pcomp(p1,nu)*pcomp(p4,mu)+2*pcomp(p2,mu)*pcomp(p3,nu)+2*pcomp(p2,mu)*pcomp(p4,nu)+2*pcomp(p2,nu)*pcomp(p3,mu)+2*pcomp(p2,nu)*pcomp(p4,mu)+2*pcomp(p3,mu)*pcomp(p4,nu)+2*pcomp(p3,nu)*pcomp(p4,mu))*Delta(p1+p2+p3+p4)


## 3. Add fermions and tensor structures

For fermions we must tell the engine more:

- the statistics are fermionic
- the field roles are `psibar`, `psi`, `scalar`, `vector`, ...
- the spinor labels must be explicit if we want open spinor-index output

This is where the direct engine already starts to look like a real Feynman-rule generator.

In [12]:
L_yukawa = {
    "coupling": yF,
    "alphas": [psibar0, psi0, phi0],
    "betas": [b1, b2, b3],
    "ps": [p1, p2, p3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "scalar"],
    "leg_roles": ["psibar", "psi", "scalar"],
    "field_spinor_indices": [a,a, None],
    "leg_spins": [s1, s2, s3],
}

V_yukawa = direct_vertex(
    **L_yukawa,
    species_map={b1: psibar0, b2: psi0, b3: phi0},
)

L_vec_current = {
    "coupling": gV * gamma_matrix(i_bar, i_psi, mu),
    "alphas": [psibar0, psi0, A0],
    "betas": [b1, b2, b3],
    "ps": [p1, p2, p3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "vector"],
    "leg_roles": ["psibar", "psi", "vector"],
    "field_index_labels": [
        {SPINOR_KIND: i_bar},
        {SPINOR_KIND: i_psi},
        {LORENTZ_KIND: mu},
    ],
    "leg_index_labels": [
        {SPINOR_KIND: i1},
        {SPINOR_KIND: i2},
        {LORENTZ_KIND: mu3},
    ],
    "leg_spins": [s1, s2, s3],
}

V_vec_current = direct_vertex(
    **L_vec_current,
    species_map={b1: psibar0, b2: psi0, b3: A0},
    simplify_gamma=True,
)

show("Yukawa vertex from direct API", V_yukawa)
show("Vector current with explicit open indices", V_vec_current)

=== Yukawa vertex from direct API ===
1𝑖*yF*g(bis(4, i1),bis(4, i2))
=== Vector current with explicit open indices ===
1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


## 4. Mix fermions and derivatives

The engine is not limited to pure scalar derivatives or pure fermion bilinears.

It can also handle mixed operators such as

$$ y_F \, \bar\psi\psi\,(\partial_\mu \phi)(\partial_
u \chi). $$

In [13]:
L_mix = {
    "coupling": yF,
    "alphas": [psibar0, psi0, phi0, chi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
    "derivative_indices": [mu, nu],
    "derivative_targets": [2, 3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "scalar", "scalar"],
    "leg_roles": ["psibar", "psi", "scalar", "scalar"],
    "field_spinor_indices": [alpha_s, alpha_s, None, None],
    "leg_spins": [s1, s2, s3, s4],
}

V_mix = direct_vertex(
    **L_mix,
    species_map={b1: psibar0, b2: psi0, b3: phi0, b4: chi0},
)

show("Mixed fermion-scalar derivative operator", V_mix)

=== Mixed fermion-scalar derivative operator ===
-1𝑖*yF*g(bis(4, i1),bis(4, i2))*pcomp(p3,mu)*pcomp(p4,nu)


## 5. First metadata objects: `Field`, `FieldOccurrence`, `ExternalLeg`, `InteractionTerm`

The direct API is useful, but the codebase really wants to move toward structured model input.

The core metadata objects are:

- `Field`: one declared particle field
- `FieldOccurrence`: one appearance of that field inside an interaction term
- `ExternalLeg`: one external state used when evaluating a vertex
- `InteractionTerm`: one monomial in the Lagrangian

The key bridge method is `InteractionTerm.to_vertex_kwargs(...)`: it translates the structured model layer into the parallel-list format expected by the engine.

In [14]:
PhiField = Field("Phi", spin=0, self_conjugate=True, symbol=phi0)
ChiField = Field("Chi", spin=0, self_conjugate=True, symbol=chi0)
PsiField = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX,),
)
GaugeField = Field("A", spin=1, self_conjugate=True, symbol=A0, indices=(LORENTZ_INDEX,))

phi_occ = PhiField.occurrence()
psibar_occ = PsiField.occurrence(conjugated=True, labels={SPINOR_KIND: alpha_s})
psi_leg = PsiField.leg(p2, spin=s2, labels={SPINOR_KIND: i2})

print("Example FieldOccurrence:", psibar_occ)
print("  role   =", psibar_occ.role)
print("  species=", psibar_occ.species)
print()
print("Example ExternalLeg:", psi_leg)
print("  role            =", psi_leg.role)
print("  effective species=", psi_leg.effective_species)

TERM_phi4 = InteractionTerm(
    coupling=lam4,
    fields=tuple(PhiField.occurrence() for _ in range(4)),
    label="lam4 * phi^4",
)
LEGS_phi4 = tuple(PhiField.leg(p, species=b) for p, b in [(p1, b1), (p2, b2), (p3, b3), (p4, b4)])

TERM_yukawa = InteractionTerm(
    coupling=yF,
    fields=(
        PsiField.occurrence(conjugated=True, labels={SPINOR_KIND: alpha_s}),
        PsiField.occurrence(labels={SPINOR_KIND: alpha_s}),
        PhiField.occurrence(),
    ),
    label="yF * psibar psi phi",
)

LEGS_yukawa = (
    PsiField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1}),
    PsiField.leg(p2, spin=s2, labels={SPINOR_KIND: i2}),
    PhiField.leg(p3, species=b3),
)

print("Keys produced by InteractionTerm.to_vertex_kwargs(...):")
pprint(sorted(TERM_yukawa.to_vertex_kwargs(LEGS_yukawa).keys()))

V_phi4_model = model_vertex(
    TERM_phi4,
    LEGS_phi4,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_yukawa_model = model_vertex(
    TERM_yukawa,
    LEGS_yukawa,
    species_map={b3: phi0},
)

show("phi^4 through the model layer", V_phi4_model)
show("Yukawa through the model layer", V_yukawa_model)

assert V_yukawa_model.expand().to_canonical_string() == V_yukawa.expand().to_canonical_string()
print("\
Direct Yukawa and model-layer Yukawa agree.")

Example FieldOccurrence: FieldOccurrence(field=Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi0, conjugate_symbol=psibar0, mass=None, quantum_numbers={}), conjugated=True, labels={'spinor': alpha_s})
  role   = FieldRole('psibar')
  species= psibar0

Example ExternalLeg: ExternalLeg(field=Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi0, conjugate_symbol=psibar0, mass=None, quantum_numbers={}), momentum=p2, conjugated=False, species=None, spin=s2, labels={'spinor': i2})
  role            = FieldRole('psi')
  effective species= psi0
Keys produced by InteractionTerm.to_vertex_kwargs(...):
['alphas',
 'b

## 6. First `Model` object

A `Model` is the top-level container for a theory.

At minimum it can already store:

- fields
- explicit interactions
- gauge groups
- later: covariant terms and gauge kinetic terms

In [15]:
ToyScalarModel = Model(
    name="Toy scalar model",
    fields=(PhiField, ChiField),
    interactions=(TERM_phi4,),
)

print(ToyScalarModel)
print()
print("find_field('Phi') ->", ToyScalarModel.find_field("Phi"))
print("first stored interaction label ->", ToyScalarModel.interactions[0].label)

Model(name='Toy scalar model', gauge_groups=(), fields=(Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), Field(name='Chi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=chi0, conjugate_symbol=None, mass=None, quantum_numbers={})), parameters=(), interactions=(InteractionTerm(coupling=lam4, fields=(FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), conjugated=False, labels={}), FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), conjugated=False, labels={}), FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=

## 7. Gauge metadata: `GaugeRepresentation` and `GaugeGroup`

To move beyond handwritten interaction terms, the code introduces gauge metadata.

- `GaugeRepresentation` tells the compiler how to build the generator tensor for one matter representation
- `GaugeGroup` defines the symmetry itself: coupling, gauge boson, structure constants, charge label, and supported matter representations

This is the basis for the gauge compiler.

In [16]:
PhiQEDField = Field(
    "PhiQED",
    spin=0,
    self_conjugate=False,
    symbol=phiQED0,
    conjugate_symbol=phiQEDdag0,
    quantum_numbers={"Q": qPhi},
)

PsiQEDField = Field(
    "PsiQED",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psiQED0,
    conjugate_symbol=psibarQED0,
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)

PhiQCDField = Field(
    "PhiQCD",
    spin=0,
    self_conjugate=False,
    symbol=phiQCD0,
    conjugate_symbol=phiQCDdag0,
    indices=(COLOR_FUND_INDEX,),
)

QuarkField = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)

GluonField = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=G0,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)

COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)

QED_GROUP = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson="A",
    charge="Q",
)

QCD_GROUP = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_QED_BASE = Model(
    name="FermionQED-minimal",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
)

MODEL_QCD_BASE = Model(
    name="QCD-minimal",
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
)

print("QCD representation seen by the quark ->", QCD_GROUP.matter_representation(QuarkField))
print("Gauge boson resolved for QED ->", MODEL_QED_BASE.gauge_boson_field(QED_GROUP))

QCD representation seen by the quark -> GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10916d7a0>, name='fundamental')
Gauge boson resolved for QED -> Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={})


## 8. Minimal gauge compiler

The **minimal gauge compiler** turns gauge metadata into ordinary `InteractionTerm` objects.

It is a structural compiler: it inserts the right charges, generators, and spectator identities, but it is not yet the convention-fixed covariant expansion of full kinetic terms.

In [17]:
compiled_qed = compile_minimal_gauge_interactions(MODEL_QED_BASE)
compiled_qcd = compile_minimal_gauge_interactions(MODEL_QCD_BASE)

print("Minimal QED terms:", labels(compiled_qed))
print("Minimal QCD terms:", labels(compiled_qcd))

LEGS_qed = (
    PsiQEDField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1}),
    PsiQEDField.leg(p2, spin=s2, labels={SPINOR_KIND: i2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}),
)

LEGS_quark_gluon = (
    QuarkField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    QuarkField.leg(p2, spin=s2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)

V_qed_min = model_vertex(compiled_qed[0], LEGS_qed, simplify_gamma=True)
V_qcd_min = model_vertex(compiled_qcd[0], LEGS_quark_gluon, simplify_gamma=True)

show("Minimal gauge compiler: fermion QED current", V_qed_min)
show("Minimal gauge compiler: quark-gluon current", V_qcd_min)

Minimal QED terms: ['U1QED: PsiQED gauge current']
Minimal QCD terms: ['SU3C: q gauge current']
=== Minimal gauge compiler: fermion QED current ===
1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
=== Minimal gauge compiler: quark-gluon current ===
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))


## 9. Convention-fixed kinetic terms and the covariant compiler

The newer physics-facing layer is the **convention-fixed compiler**.

Instead of hand-declaring each interaction, we declare standard kinetic structures such as:

- `DiracKineticTerm(field=...)` for $\bar\psi i \gamma^\mu D_\mu \psi$
- `ComplexScalarKineticTerm(field=...)` for $(D_\mu \phi)^\dagger (D^\mu \phi)$

The compiler then expands these according to the fixed conventions of the project.

In [18]:
MODEL_QED_FERMION_COV = Model(
    name="FermionQED-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
    covariant_terms=(DiracKineticTerm(field=PsiQEDField),),
)

compiled_qed_cov = compile_covariant_terms(MODEL_QED_FERMION_COV)
print("Covariant fermion-QED terms:", labels(compiled_qed_cov))

V_qed_cov = model_vertex(compiled_qed_cov[0], LEGS_qed, simplify_gamma=True)

show("Convention-fixed Dirac kinetic term expanded to QED current", V_qed_cov)

MODEL_SCALAR_QED_COV = Model(
    name="ScalarQED-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(PhiQEDField, GaugeField),
    covariant_terms=(ComplexScalarKineticTerm(field=PhiQEDField),),
)

compiled_sqed = compile_covariant_terms(MODEL_SCALAR_QED_COV)
print("Covariant scalar-QED terms:", labels(compiled_sqed))

LEGS_sqed_current = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)

LEGS_sqed_contact = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

sqed_species_3 = {b1: phiQEDdag0, b2: phiQED0, b3: A0}
sqed_species_4 = {b1: phiQEDdag0, b2: phiQED0, b3: A0, b4: A0}

V_sqed_current = (
    model_vertex(compiled_sqed[0], LEGS_sqed_current, species_map=sqed_species_3)
    + model_vertex(compiled_sqed[1], LEGS_sqed_current, species_map=sqed_species_3)
)

V_sqed_contact = model_vertex(compiled_sqed[2], LEGS_sqed_contact, species_map=sqed_species_4)

show("Complex-scalar kinetic term: combined current", V_sqed_current)
show("Complex-scalar kinetic term: contact term", V_sqed_contact)

Covariant fermion-QED terms: ['i PsiQEDbar gamma^mu D_mu PsiQED']
=== Convention-fixed Dirac kinetic term expanded to QED current ===
-1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
Covariant scalar-QED terms: ['(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (+)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (-)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar contact']
=== Complex-scalar kinetic term: combined current ===
-1𝑖*eQED*qPhi*pcomp(p1,mu)+1𝑖*eQED*qPhi*pcomp(p2,mu)
=== Complex-scalar kinetic term: contact term ===
2𝑖*eQED^2*qPhi^2*g(mink(4, mu3),mink(4, mu4))


## 10. Pure-gauge sector and Yang-Mills self-interactions

The latest implemented layer is the pure-gauge kinetic sector.

A single `GaugeKineticTerm(...)` can now generate:

- the gauge bilinear
- the Yang-Mills cubic vertex
- the Yang-Mills quartic vertex

for the covered non-abelian case.

In [19]:
MODEL_QCD_GAUGE_COV = Model(
    name="QCDGauge-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(GluonField,),
    gauge_kinetic_terms=(GaugeKineticTerm(gauge_group=QCD_GROUP),),
)

compiled_ym = compile_covariant_terms(MODEL_QCD_GAUGE_COV)

print("Number of generated pure-gauge interactions:", len(compiled_ym))
print("Generated labels:")
for label in labels(compiled_ym):
    print(" -", label)

LEGS_three_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
)

V_ym_cubic = model_vertex(
    compiled_ym[2],
    LEGS_three_gluon,
    species_map={b1: G0, b2: G0, b3: G0},
)

V_ym_cubic = simplify_gamma_chain(V_ym_cubic)

show("Yang-Mills cubic vertex from GaugeKineticTerm", V_ym_cubic)

Number of generated pure-gauge interactions: 4
Generated labels:
 - -1/4 SU3C field strength squared SU3C: gauge kinetic bilinear (metric)
 - -1/4 SU3C field strength squared SU3C: gauge kinetic bilinear (cross)
 - -1/4 SU3C field strength squared SU3C: Yang-Mills cubic
 - -1/4 SU3C field strength squared SU3C: Yang-Mills quartic
=== Yang-Mills cubic vertex from GaugeKineticTerm ===
1𝑖*(-1𝑖*gS*f(coad(8, a3),coad(8, a4),coad(8, a5))*pcomp(p1,rho_G_SU3C_cubic)-1𝑖*gS*f(coad(8, a5),coad(8, a4),coad(8, a3))*pcomp(p3,rho_G_SU3C_cubic))*g(mink(4, mu),mink(4, rho))*g(mink(4, nu),mink(4, rho_G_SU3C_cubic))+1𝑖*(-1𝑖*gS*f(coad(8, a3),coad(8, a5),coad(8, a4))*pcomp(p1,rho_G_SU3C_cubic)-1𝑖*gS*f(coad(8, a4),coad(8, a5),coad(8, a3))*pcomp(p2,rho_G_SU3C_cubic))*g(mink(4, mu),mink(4, nu))*g(mink(4, rho),mink(4, rho_G_SU3C_cubic))+1𝑖*(-1𝑖*gS*f(coad(8, a4),coad(8, a3),coad(8, a5))*pcomp(p2,rho_G_SU3C_cubic)-1𝑖*gS*f(coad(8, a5),coad(8, a3),coad(8, a4))*pcomp(p3,rho_G_SU3C_cubic))*g(mink(4, mu),mink(4, rho_

## 11. What this code can currently do

By the end of this notebook we have seen the full implemented workflow:

1. **Direct engine usage** for scalar and fermion interaction terms
2. **Derivative handling** through momentum factors
3. **Open spinor and Lorentz index handling** through explicit labels
4. **Structured model-layer declarations** using `Field`, `ExternalLeg`, `InteractionTerm`, and `Model`
5. **Minimal gauge compilation** from metadata
6. **Convention-fixed covariant compilation** for matter kinetic terms
7. **Pure-gauge field-strength compilation** up to Yang-Mills 3- and 4-point vertices

### Current codebase strengths

- one central engine drives both direct and model-based workflows
- the tensor vocabulary is explicit and reusable
- the gauge compiler is already able to derive standard matter and pure-gauge interactions from metadata
- the code is organized by layer rather than by isolated examples

### Current limitations

- multi-fermion tensor support is still narrower than a full FeynRules-like system
- repeated identical index kinds are still a structural weak point in the model/compiler boundary
- gauge fixing, ghosts, and BFM-specific splitting are not yet implemented
- much of the validation still lives in `src/examples.py` rather than a dedicated test suite

### Recommended next reading in the source tree

- `src/model.py`: the data model
- `src/model_symbolica.py`: the engine
- `src/gauge_compiler.py`: the compiler path from model metadata to explicit interaction terms
- `src/spenso_structures.py`: the tensor building blocks

This notebook is meant to make those files much easier to read.